# 第 6 章 · 合并访存：一个 warp 一次读显存，要搬几趟

本章回答：32 个线程同时读显存，硬件是一个一个读，还是打包读？为什么"相邻线程读相邻地址"这么重要？跨步读、错位读各浪费多少？矩阵按行读和按列读差多少？

配套讲义：仓库里 `cuda-programming/notes/lessons/06-coalescing.md`（讲解、动画、练习都在讲义里）。对应原文：PG §2.3.4.1 Coalesced Global Memory Access（`materials/text/cuda-programming-guide.txt` 第 3743–3929 行，含矩阵转置例子）；Best Practices §10.2.1（`text/cuda-c-best-practices-guide.txt` 第 1400–1574 行）；GPU MODE L8 性能清单（`text/gpu-mode-lecture-08-cuda-performance-checklist.txt`）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 ①–④ | 箱数 = 不同的 ⌊地址 / 32⌋ 的个数 | `sectors(warp_addrs(stride, offset))` | 4、5、8、32 箱；100%、80%、50%、12.5% |
| 第二步 跨步公式 | $4s$ 箱、利用率 $1/s$ | `[sectors(warp_addrs(s)) for s in range(1, 9)]` | [4, 8, …, 32] |
| 第二步 顺序打乱 | 比的是用到 ÷ 搬来 | `sectors(rng.permutation(warp_addrs(1)))` | 仍是 4 箱 |
| 第三步 按行 / 按列 | `r * N + c` | `row_major_addrs(3, tx, 64)` / `row_major_addrs(tx, 3, 64)` | 4 箱 vs 32 箱 |
| 第四步 float4 | 每线程 16 字节 | `warp_addrs(1, 0, word=16)` | 512 B、16 箱、100% |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `SECTOR`**（来自 `ch06_coalescing.py`）

In [ ]:
SECTOR = 32

**常量 `WARP`**（来自 `ch06_coalescing.py`）

In [ ]:
WARP = 32

**函数 `row_major_addrs`**（来自 `ch06_coalescing.py`）：二维数组 A[r][c] 按行优先存：位置 = r * N + c。

In [ ]:
def row_major_addrs(rows: np.ndarray, cols: np.ndarray, n_cols: int, word: int = 4) -> np.ndarray:
    """二维数组 A[r][c] 按行优先存：位置 = r * N + c。"""
    return (rows * n_cols + cols) * word

**函数 `sectors`**（来自 `ch06_coalescing.py`）：32 个地址落在几箱：不同的 ⌊地址 / 32⌋ 的个数。

In [ ]:
def sectors(byte_addrs: np.ndarray) -> int:
    """32 个地址落在几箱：不同的 ⌊地址 / 32⌋ 的个数。"""
    return len(np.unique(np.asarray(byte_addrs) // SECTOR))

**函数 `utilization`**（来自 `ch06_coalescing.py`）：利用率 = 用到的字节 ÷ 搬来的字节。

In [ ]:
def utilization(byte_addrs: np.ndarray, word: int = 4) -> float:
    """利用率 = 用到的字节 ÷ 搬来的字节。"""
    return len(byte_addrs) * word / (sectors(byte_addrs) * SECTOR)

**函数 `warp_addrs`**（来自 `ch06_coalescing.py`）：线程 t 读 a[t * stride + offset]，每个元素 word 字节；返回 32 个字节地址（数组从 0 号地址开始，已对齐）。

In [ ]:
def warp_addrs(stride: int, offset: int = 0, word: int = 4) -> np.ndarray:
    """线程 t 读 a[t * stride + offset]，每个元素 word 字节；返回 32 个字节地址（数组从 0 号地址开始，已对齐）。"""
    t = np.arange(WARP)
    return (t * stride + offset) * word

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 四种情况（每个线程读一个 4 字节 float）")
cases = [("① a[i]", 1, 0), ("② a[i + 1]", 1, 1), ("③ a[2 * i]", 2, 0), ("④ a[8 * i]", 8, 0), ("④ a[32 * i]", 32, 0)]
for name, s, o in cases:
    ad = warp_addrs(s, o)
    print(f"  {name:<12} {sectors(ad):2d} 箱，搬 {sectors(ad) * SECTOR:4d} B，利用率 {utilization(ad):.1%}")

In [ ]:
print("\n第二步 跨步 s ≤ 8 时箱数 = 4s，利用率 = 1/s")
print("  s = 1..8 的箱数：", [sectors(warp_addrs(s)) for s in range(1, 9)])

In [ ]:
print("\n第二步 顺序打乱没关系")
rng = np.random.default_rng(0)
print("  把 a[i] 的 32 个地址随机打乱后仍是", sectors(rng.permutation(warp_addrs(1))), "箱")

In [ ]:
print("\n第三步 二维数组 N = 64：warp 里 threadIdx.x = 0..31 对应 col，row 固定为 3")
tx = np.arange(WARP)
print("  in[row * N + col]（按行读）：", sectors(row_major_addrs(3, tx, 64)), "箱")
print("  in[col * N + row]（按列读）：", sectors(row_major_addrs(tx, 3, 64)), "箱")

In [ ]:
print("\n第四步 float4：每个线程一次读 16 字节")
ad = warp_addrs(1, 0, word=16)
print(f"  一条指令搬 {WARP * 16} B，{sectors(ad)} 箱，利用率 {utilization(ad, word=16):.0%}；指令数是 1/4")

## 3. 代码里才看得到的

1. **地址假设从 0 开始**：`warp_addrs` 默认数组起点对齐到 32 字节（`cudaMalloc` 至少 256 字节对齐），所以 `a[i]` 正好 4 箱；`offset = 1` 就是错位的情形。
2. **这里只数箱，不模拟缓存**：讲义说错位读在 V100 上实测只损失约 1/10，因为邻居 warp 会命中 L1 / L2；代码按"每个 warp 单独搬"算，得到的是 4/5，是最坏情况。
3. **`np.unique` 天然忽略顺序**：所以打乱 32 个地址箱数不变，和 Best Practices §10.2.1.1 说的一样。

## 4. 自己改一改

- 把 `warp_addrs(1, 1)` 的偏移换成 2、7、8，看什么时候又回到 4 箱。
- 把第三步的 N 从 64 改成 8、16，看按列读要几箱（N × 4 字节小于 32 时会有几个线程挤进同一箱）。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 显存按箱发货，一箱 32 字节；
> warp 一起下单，同箱只运一次；
> 相邻线程读相邻地址：4 箱满载；跨步 s：利用率 1/s；隔 32 字节以上：一人一箱，八分之一；
> threadIdx.x 对最后一维；转置靠 shared 中转；一次读 float4，指令省四倍。